# directory-pipeline: library cookbook

Worked examples of using pipeline pieces directly from Python — no CLI required.
Everything imported here comes from `pipeline.api`, the supported public surface
(anything not re-exported there is internal and may change without notice).

Docs: [usage-examples.md §9](../docs/usage-examples.md) · API: [`pipeline/api.py`](../pipeline/api.py)


In [ ]:
# Colab setup — skip locally (use the repo venv kernel after `uv sync` instead)
!git clone https://github.com/hadro/directory-pipeline.git
%cd directory-pipeline
!pip install -q -e .


## 1. Walk any public IIIF manifest (v2 or v3)

Works standalone — no pipeline run, no API key. `iter_canvases` normalizes
Presentation v2 and v3 into one dict shape; `image_url` builds a Image API
download URL at any width.


In [ ]:
import json, urllib.request
from pipeline.api import iter_canvases, image_url, manifest_version

MANIFEST_URL = "https://www.loc.gov/item/01015253/manifest.json"  # 1857 Brooklyn city directory
manifest = json.load(urllib.request.urlopen(MANIFEST_URL))
print("IIIF Presentation version:", manifest_version(manifest))

for canvas in list(iter_canvases(manifest))[:5]:
    print(canvas["image_id"], "→", image_url(canvas["service_id"], width=512))


## 2. Find what a pipeline run produced

`pipeline_state.json` records which model a run used; when it's missing,
`discover_ocr_slug` falls back to parsing output filenames.


In [ ]:
import csv
from pathlib import Path
from pipeline.api import get_ocr_model, get_ner_model, discover_ocr_slug, read_state

vol = Path("output/my_volume")          # ← point at one of your volumes
print(read_state(vol))
model = get_ner_model(vol) or get_ocr_model(vol) or discover_ocr_slug(vol)
print("model slug:", model)

entries_csv = next(vol.rglob(f"entries_{model}*.csv"))
with entries_csv.open(encoding="utf-8") as fh:
    entries = list(csv.DictReader(fh))
print(len(entries), "entries —", entries[0] if entries else "none")


## 3. Parse Surya OCR output

`parse_surya` returns the page bbox, the detected lines (filtered by a
confidence threshold), and the page-level median confidence — handy for
spotting low-quality scans before paying for Gemini OCR.


In [ ]:
from pipeline.api import parse_surya

for surya_json in sorted(vol.rglob("*_surya.json"))[:10]:
    page_bbox, lines, median_conf = parse_surya(surya_json)
    print(f"{surya_json.name[:40]:40s} lines={len(lines):3d} median_conf={median_conf}")


## 4. Clean and merge entries CSVs

The same normalization/dedup/category-inference pass that
`pipeline postprocess` runs, as a function call.


In [ ]:
from pipeline.api import process_csv, combine_volumes

fixed = entries_csv.with_stem(entries_csv.stem + "_fixed")
stats = process_csv(entries_csv, fixed, infer_categories=True)
print(stats)

# Merge every volume's *_fixed.csv in a collection directory:
# combine_volumes(Path("output/my_collection"), Path("output/my_collection/combined.csv"))


## 5. Call Gemini with the pipeline's retry behavior

`get_client` reads `GEMINI_API_KEY` from the environment (or a `.env` file);
`generate_with_retry` adds the exponential backoff on 429/503 the pipeline
uses everywhere. **This cell makes a paid API call.**


In [ ]:
from google.genai.types import GenerateContentConfig
from pipeline.api import DEFAULT_OCR_MODEL, get_client, generate_with_retry

client = get_client()
response = generate_with_retry(
    client,
    model=DEFAULT_OCR_MODEL,
    config=GenerateContentConfig(temperature=0.0),
    contents="In one sentence: what is a city directory?",
)
print(response.text)
